In [1]:
import sys
import importlib
sys.path.append("/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/")
import python.utils as ut

import os
import numpy as np
import arviz as az
from numpy.polynomial.legendre import legvander
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
import json
import glob
from scipy.stats import norm, cauchy, mode, t
from cmdstanpy import from_csv
import seaborn as sns
from tqdm import tqdm
import corner
import json

plt.style.use('seaborn-v0_8')

In [2]:
ABS_DIR = "/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/"
L_desired = 24
prior_path = ABS_DIR+f"data/json/priors_L{L_desired}.json"

In [3]:
prior_dict = dict(
    # --- transition priors ---
    alpha_clean       = [10.0, 1.0, 1.0],        # clean -> {clean, rising, blip}
    alpha_rising      = [10.0, 5.0, 1.0],        # rising -> {rising, decay, blip}
    alpha_decay       = [5.0, 1.0, 10.0, 1.0],   # decay  -> {clean, rising, decay, blip}
    alpha_blip        = [10.0, 1.0, 1.0, 1.0],   # blip   -> {clean, rising, decay, blip}

    # --- dynamic parameters ---
    rr_log_mu         = 0.0,        # lognormal mean for rate_rising (exp(0)=1)
    rr_log_sigma      = 1.0,        # wide
    rd_alpha          = 2.0,        # beta(2,2) near-uniform
    rd_beta           = 2.0,

    # --- noise variance ---
    sig_log_mu        = 0.0,        # mean of log(sigma)
    sig_log_sigma     = 2.0,        # covers sigma ~ [0.05, 50]

    # --- blip emission ---
    mu_blip_mean      = 0.0,        # centered
    mu_blip_sd        = 10.0,       # very wide
    k_blip_log_mu     = 0.0,        # lognormal mean for k_blip
    k_blip_log_sigma  = 2.0,        # broad spread

    # --- legendre hyperparameters (lenient, scale-invariant) ---
    mu_X_mean         = [0.0] * L_desired,              # zero-centered
    mu_X_sd           = [5.0] * L_desired,              # wide (allows large coeffs)
    alpha_X_log_mu    = [0.0] * L_desired,              # lognormal mean
    alpha_X_log_sigma = [1.0] * L_desired,              # wide dispersion
    beta_X_log_mu    = float(np.log(2.0)),   # median(beta_X) = 2
    beta_X_log_sigma = 1.0
)

In [4]:
with open(
    prior_path,
    "w"
) as f:
    json.dump(prior_dict, f, indent=2)